In [27]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [28]:
@tool
def get_flight_cost(origin: str, destination: str) -> str:
    """Estimate round-trip flight cost per person between an origin and destination."""

    route_prices = {
        ("delhi", "goa"): 8500,
        ("mumbai", "goa"): 4500,
        ("delhi", "mumbai"): 6000,
        ("mumbai", "delhi"): 6000
    }

    route = (origin.strip().lower(), destination.strip().lower())
    reverse_route = (route[1], route[0])

    price = route_prices.get(route, route_prices.get(reverse_route))

    if price is None:
        price = 10000  # Demonstration fallback estimate

    return (
        f"Estimated round-trip flight cost per person from "
        f"{origin.title()} to {destination.title()}: ₹{price:,.2f}. "
        "This is a sample estimate, not a live fare."
    )


print(get_flight_cost.invoke({
    "origin": "Delhi",
    "destination": "Goa"
}))

Estimated round-trip flight cost per person from Delhi to Goa: ₹8,500.00. This is a sample estimate, not a live fare.


In [29]:
@tool
def get_hotel_cost(city: str, nights: int) -> str:
    """Estimate hotel cost for a given city and number of nights."""

    if nights <= 0:
        raise ValueError("Number of nights must be positive.")

    nightly_rates = {
        "goa": 2500,
        "mumbai": 3500,
        "delhi": 3000,
        "pune": 2800
    }

    nightly_rate = nightly_rates.get(city.strip().lower(), 3000)
    total_cost = nightly_rate * nights

    return (
        f"Estimated hotel cost in {city.title()} for {nights} nights: "
        f"₹{total_cost:,.2f} "
        f"(sample rate ₹{nightly_rate:,.2f} per night). "
        "This is a sample estimate, not a live hotel price."
    )


print(get_hotel_cost.invoke({
    "city": "Goa",
    "nights": 3
}))

Estimated hotel cost in Goa for 3 nights: ₹7,500.00 (sample rate ₹2,500.00 per night). This is a sample estimate, not a live hotel price.


In [30]:
@tool
def get_food_budget(city: str, days: int) -> str:
    """Estimate food expenses for one person for a specified number of days."""

    if days <= 0:
        raise ValueError("Number of days must be positive.")

    daily_budgets = {
        "goa": 800,
        "mumbai": 1000,
        "delhi": 750,
        "pune": 700
    }

    daily_budget = daily_budgets.get(city.strip().lower(), 800)
    total_budget = daily_budget * days

    return (
        f"Estimated food budget in {city.title()} for {days} days: "
        f"₹{total_budget:,.2f} "
        f"(sample budget ₹{daily_budget:,.2f} per day). "
        "This is a sample estimate, not a live price."
    )


print(get_food_budget.invoke({
    "city": "Goa",
    "days": 4
}))

Estimated food budget in Goa for 4 days: ₹3,200.00 (sample budget ₹800.00 per day). This is a sample estimate, not a live price.


In [31]:
tools = [
    get_flight_cost,
    get_hotel_cost,
    get_food_budget
]

tool_map = {
    "get_flight_cost": get_flight_cost,
    "get_hotel_cost": get_hotel_cost,
    "get_food_budget": get_food_budget
}

llm_with_tools = llm.bind_tools(tools)

print("All three tools bound in one call:")

for item in tools:
    print("-", item.name)

All three tools bound in one call:
- get_flight_cost
- get_hotel_cost
- get_food_budget


In [37]:
user_query = """
Plan a 3-night trip for one person from Delhi to Goa.

You MUST call each of these three tools:
1. get_flight_cost with origin Delhi and destination Goa.
2. get_hotel_cost with city Goa and nights 3.
3. get_food_budget with city Goa and days 4.

Call all three tools. Do not answer until all three cost estimates
have been retrieved.
"""

messages = [HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages)

print("Model-suggested tool calls:")

for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

called_tools = {call["name"] for call in response.tool_calls}

print("\nTools requested:", called_tools)

Model-suggested tool calls:
Tool: get_flight_cost
Arguments: {'destination': 'Goa', 'origin': 'Delhi'}

Tools requested: {'get_flight_cost'}


In [38]:
messages.append(response)

tool_results = []
executed_tools = set()

# Execute the model-suggested tool calls first
for call in response.tool_calls:
    tool_name = call["name"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    result = tool_map[tool_name].invoke(call["args"])
    tool_results.append((tool_name, result))
    executed_tools.add(tool_name)

    print(f"\nModel-suggested tool executed: {tool_name}")
    print("Result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

# Execute any required tool the model omitted
required_calls = [
    ("get_flight_cost", {
        "origin": "Delhi",
        "destination": "Goa"
    }),
    ("get_hotel_cost", {
        "city": "Goa",
        "nights": 3
    }),
    ("get_food_budget", {
        "city": "Goa",
        "days": 4
    })
]

for tool_name, tool_args in required_calls:
    if tool_name not in executed_tools:
        result = tool_map[tool_name].invoke(tool_args)
        tool_results.append((tool_name, result))
        executed_tools.add(tool_name)

        print(f"\nRequired tool executed separately: {tool_name}")
        print("Result:", result)

print("\nAll required tools executed.")
print("Executed tools:", executed_tools)


Model-suggested tool executed: get_flight_cost
Result: Estimated round-trip flight cost per person from Delhi to Goa: ₹8,500.00. This is a sample estimate, not a live fare.

Required tool executed separately: get_hotel_cost
Result: Estimated hotel cost in Goa for 3 nights: ₹7,500.00 (sample rate ₹2,500.00 per night). This is a sample estimate, not a live hotel price.

Required tool executed separately: get_food_budget
Result: Estimated food budget in Goa for 4 days: ₹3,200.00 (sample budget ₹800.00 per day). This is a sample estimate, not a live price.

All required tools executed.
Executed tools: {'get_flight_cost', 'get_food_budget', 'get_hotel_cost'}


In [39]:
tool_results_text = "\n\n".join(
    f"{name}:\n{result}"
    for name, result in tool_results
)

final_prompt = HumanMessage(
    content=(
        "Prepare a combined trip cost estimate using the following "
        "three tool results. Include flights, hotel, food, and the total. "
        "Use the exact returned prices. Do not invent or alter prices. "
        "Label all figures as sample estimates.\n\n"
        f"{tool_results_text}"
    )
)

final_response = llm.invoke([final_prompt])

print("Final trip cost estimate:")
print(final_response.content)

Final trip cost estimate:
**Combined Trip Cost Estimate (Sample Estimates)**  

| Item | Cost (₹) | Note |
|------|----------|------|
| **Round‑trip flight (Delhi → Goa)** | **8,500.00** | Sample estimate per person |
| **Hotel (3 nights)** | **7,500.00** | Sample estimate (₹2,500.00 per night) |
| **Food (4 days)** | **3,200.00** | Sample estimate (₹800.00 per day) |
| **Total (flight + hotel + food)** | **19,200.00** | Sample estimate for a single traveler |

All figures are taken directly from the tool outputs and are labeled as **sample estimates**. No prices have been altered or invented.


In [40]:
required_tools = {
    "get_flight_cost",
    "get_hotel_cost",
    "get_food_budget"
}

called_tools = {
    call["name"] for call in response.tool_calls
}

executed_tools = {
    name for name, result in tool_results
}

print("Model-suggested tools:", called_tools)
print("Executed tools:", executed_tools)

assert required_tools.issubset(executed_tools), (
    "One or more required tools were not executed."
)

assert len(tool_results) == 3
assert final_response.content.strip()

print("All three tools executed successfully.")
print("Final answer generated.")

Model-suggested tools: {'get_flight_cost'}
Executed tools: {'get_flight_cost', 'get_food_budget', 'get_hotel_cost'}
All three tools executed successfully.
Final answer generated.
